In [1]:
import os

In [2]:
%pwd

'd:\\End_to_End_ML_project_for_Loan_Default_Prediction\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\End_to_End_ML_project_for_Loan_Default_Prediction'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    input_file_path: Path
    train_file_path: Path
    validation_file_path: Path
    test_file_path: Path
    test_size: float
    validation_size: float
    random_state: int
    target_column: str
    split_artifacts_dir: Path

In [6]:
from src.loan_default_prediction.constants import *
from loan_default_prediction.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(self, config_filepath = CONFIG_FILE_PATH, params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            split_artifacts_dir=config.split_artifacts_dir,
            input_file_path=config.input_file_path,
            train_file_path=config.train_file_path,
            validation_file_path=config.validation_file_path,
            test_file_path=config.test_file_path,
            test_size=float(config.test_size),
            validation_size=float(config.validation_size),
            random_state=int(config.random_state),
            target_column=config.target_column
        )

        return data_transformation_config    

In [8]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from loan_default_prediction import logger

In [9]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config

    def load_processed_data(self) -> pd.DataFrame:
        if not os.path.exists(self.config.input_file_path):
            raise FileNotFoundError(f"Processed data not found: {self.config.input_file_path}")

        logger.info(f"Loading processed data from: {self.config.input_file_path}")
        data = pd.read_csv(self.config.input_file_path)
        logger.info(f"Processed data shape: {data.shape}")
        return data

    def split_data(self, data: pd.DataFrame):
        target_col = self.config.target_column
        if target_col not in data.columns:
            raise KeyError(f"Target column '{target_col}' not found in data")

        logger.info("Creating stratified train/validation/test split...")
        X = data.drop(columns=[target_col])
        y = data[target_col]

        # First split out the test set
        X_train_val, X_test, y_train_val, y_test = train_test_split(
            X,
            y,
            test_size=self.config.test_size,
            random_state=self.config.random_state,
            stratify=y if len(y.unique()) > 1 else None
        )

        # Then split the remaining data into training and validation sets
        relative_val_size = self.config.validation_size / (1.0 - self.config.test_size)
        X_train, X_val, y_train, y_val = train_test_split(
            X_train_val,
            y_train_val,
            test_size=relative_val_size,
            random_state=self.config.random_state,
            stratify=y_train_val if len(y_train_val.unique()) > 1 else None
        )

        train_df = pd.concat([X_train.reset_index(drop=True), y_train.reset_index(drop=True)], axis=1)
        val_df = pd.concat([X_val.reset_index(drop=True), y_val.reset_index(drop=True)], axis=1)
        test_df = pd.concat([X_test.reset_index(drop=True), y_test.reset_index(drop=True)], axis=1)

        logger.info(f"Train shape: {train_df.shape}, Validation shape: {val_df.shape}, Test shape: {test_df.shape}")
        return train_df, val_df, test_df

    def apply_smote(self, train_df: pd.DataFrame):
        target_col = self.config.target_column
        if target_col not in train_df.columns:
            raise KeyError(f"Target column '{target_col}' not found in training data")

        logger.info("Applying SMOTE to the training set...")
        X_train = train_df.drop(columns=[target_col])
        y_train = train_df[target_col]

        smote = SMOTE(random_state=self.config.random_state)
        X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

        train_resampled = pd.concat([pd.DataFrame(X_resampled, columns=X_train.columns),
                                     pd.Series(y_resampled, name=target_col)], axis=1)

        logger.info(f"SMOTE applied. Training shape after resampling: {train_resampled.shape}")
        return train_resampled

    def save_split_data(self, train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame) -> None:
        os.makedirs(self.config.split_artifacts_dir, exist_ok=True)

        train_path = self.config.train_file_path
        val_path = self.config.validation_file_path
        test_path = self.config.test_file_path

        logger.info(f"Saving train data to: {train_path}")
        train_df.to_csv(train_path, index=False)

        logger.info(f"Saving validation data to: {val_path}")
        val_df.to_csv(val_path, index=False)

        logger.info(f"Saving test data to: {test_path}")
        test_df.to_csv(test_path, index=False)

        logger.info(f"All split datasets saved to: {self.config.split_artifacts_dir}")

    def initiate_data_transformation(self) -> bool:
        try:
            data = self.load_processed_data()
            train_df, val_df, test_df = self.split_data(data)
            train_df = self.apply_smote(train_df)
            self.save_split_data(train_df, val_df, test_df)
            logger.info("Data transformation completed successfully")
            return True
        except Exception as e:
            logger.error(f"Error during data transformation: {str(e)}")
            raise e

In [10]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.initiate_data_transformation()
except Exception as e:
    logger.error(f"Data transformation failed: {str(e)}")
    raise e    

[2026-05-03 02:36:02,658: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-03 02:36:02,661: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-03 02:36:02,663: INFO: common: created directory at: artifacts]
[2026-05-03 02:36:02,666: INFO: common: created directory at: artifacts/data_transformation]
[2026-05-03 02:36:02,667: INFO: 1821947932: Loading processed data from: artifacts/data_preprocessing/processed_data.csv]
[2026-05-03 02:36:05,120: INFO: 1821947932: Processed data shape: (255347, 40)]
[2026-05-03 02:36:05,124: INFO: 1821947932: Creating stratified train/validation/test split...]
[2026-05-03 02:36:05,869: INFO: 1821947932: Train shape: (178742, 40), Validation shape: (38302, 40), Test shape: (38303, 40)]
[2026-05-03 02:36:05,907: INFO: 1821947932: Applying SMOTE to the training set...]
[2026-05-03 02:36:08,142: INFO: 1821947932: SMOTE applied. Training shape after resampling: (315970, 40)]
[2026-05-03 02:36:08,179: INFO: 1821947932: S